# Задача 8. Рекомендательная система на основе матричного разложения

- данные: MovieLens 100k
- модель: latent factor model через эмбеддинги пользователей и объектов
- baseline: most popular, user-based CF, item-based CF
- метрики: RMSE, Precision@K, Recall@K, MAP@K, NDCG@K, Coverage

In [1]:
import math
import random
import urllib.request
import zipfile
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics.pairwise import cosine_similarity
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
cfg = SimpleNamespace(
    data_dir="data/ml-100k",
    k=10,
    n_factors_list=(16, 32, 64),
    reg_list=(1e-4, 1e-3),
    num_epochs=15,
    batch_size=2048,
    learning_rate=5e-3,
    log_root="runs/task8",
)
data_dir = Path(cfg.data_dir)
archive_path = data_dir.parent / "ml-100k.zip"

if not data_dir.exists():
    data_dir.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        archive_path,
    )
    with zipfile.ZipFile(archive_path, "r") as zip_file:
        zip_file.extractall(data_dir.parent)

ratings = pd.read_csv(
    data_dir / "u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
)
movies = pd.read_csv(
    data_dir / "u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["item_id", "title"],
)
ratings.head()


,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [3]:
def split_by_user_timestamp(ratings_df: pd.DataFrame):
    ratings_df = ratings_df.sort_values(["user_id", "timestamp"]).copy()
    train_parts = []
    val_parts = []
    test_parts = []
    for _, user_part in ratings_df.groupby("user_id"):
        if len(user_part) < 5:
            continue
        train_parts.append(user_part.iloc[:-2])
        val_parts.append(user_part.iloc[-2:-1])
        test_parts.append(user_part.iloc[-1:])
    train_df = pd.concat(train_parts).reset_index(drop=True)
    val_df = pd.concat(val_parts).reset_index(drop=True)
    test_df = pd.concat(test_parts).reset_index(drop=True)
    return train_df, val_df, test_df


train_df, val_df, test_df = split_by_user_timestamp(ratings)

user_ids = sorted(ratings["user_id"].unique())
item_ids = sorted(ratings["item_id"].unique())
user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
item_to_idx = {item_id: idx for idx, item_id in enumerate(item_ids)}
idx_to_item = {idx: item_id for item_id, idx in item_to_idx.items()}
item_id_to_title = dict(zip(movies["item_id"], movies["title"]))


def add_indices(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame["user_idx"] = frame["user_id"].map(user_to_idx)
    frame["item_idx"] = frame["item_id"].map(item_to_idx)
    return frame


train_df = add_indices(train_df)
val_df = add_indices(val_df)
test_df = add_indices(test_df)

n_users = len(user_to_idx)
n_items = len(item_to_idx)

train_matrix = np.zeros((n_users, n_items), dtype=np.float32)
for row in train_df.itertuples(index=False):
    train_matrix[row.user_idx, row.item_idx] = row.rating

train_seen = train_df.groupby("user_idx")["item_idx"].apply(set).to_dict()
val_positive = val_df[val_df["rating"] >= 4].groupby("user_idx")["item_idx"].apply(set).to_dict()
test_positive = test_df[test_df["rating"] >= 4].groupby("user_idx")["item_idx"].apply(set).to_dict()

print(train_df.shape, val_df.shape, test_df.shape)


(98114, 6) (943, 6) (943, 6)


In [4]:
def rmse_from_predictions(frame: pd.DataFrame, score_fn) -> float:
    predictions = []
    targets = []
    for row in frame.itertuples(index=False):
        predictions.append(score_fn(row.user_idx, row.item_idx))
        targets.append(row.rating)
    predictions = np.array(predictions, dtype=np.float32)
    targets = np.array(targets, dtype=np.float32)
    return float(np.sqrt(np.mean((predictions - targets) ** 2)))


def ranking_metrics(score_all_items_fn, positives_dict, seen_dict, top_k: int = 10) -> dict:
    precisions = []
    recalls = []
    average_precisions = []
    ndcgs = []
    recommended_items = set()

    for user_idx, positives in positives_dict.items():
        if not positives:
            continue
        scores = score_all_items_fn(user_idx).copy()
        seen = seen_dict.get(user_idx, set())
        if seen:
            scores[list(seen)] = -1e9
        top_items = np.argsort(scores)[-top_k:][::-1]
        recommended_items.update(top_items.tolist())

        hits = [1 if item in positives else 0 for item in top_items]
        hit_count = sum(hits)
        precisions.append(hit_count / top_k)
        recalls.append(hit_count / len(positives))

        running_hits = 0
        ap = 0.0
        dcg = 0.0
        for rank, hit in enumerate(hits, start=1):
            if hit:
                running_hits += 1
                ap += running_hits / rank
                dcg += 1.0 / math.log2(rank + 1)
        ideal_hits = min(len(positives), top_k)
        idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
        average_precisions.append(ap / max(1, min(len(positives), top_k)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return {
        f"precision@{top_k}": float(np.mean(precisions)),
        f"recall@{top_k}": float(np.mean(recalls)),
        f"map@{top_k}": float(np.mean(average_precisions)),
        f"ndcg@{top_k}": float(np.mean(ndcgs)),
        "coverage": float(len(recommended_items) / n_items),
    }


def fit_most_popular(train_frame: pd.DataFrame, n_items: int) -> dict:
    item_stats = train_frame.groupby("item_idx")["rating"].agg(["mean", "count"])
    item_mean = np.full(n_items, train_frame["rating"].mean(), dtype=np.float32)
    popularity = np.zeros(n_items, dtype=np.float32)
    item_mean[item_stats.index.to_numpy()] = item_stats["mean"].to_numpy(dtype=np.float32)
    popularity[item_stats.index.to_numpy()] = item_stats["count"].to_numpy(dtype=np.float32)
    return {"item_mean": item_mean, "popularity": popularity}


def popular_predict(model: dict, user_idx: int, item_idx: int) -> float:
    return float(model["item_mean"][item_idx])


def popular_score_all_items(model: dict, user_idx: int) -> np.ndarray:
    return model["popularity"].copy()


def fit_user_cf(matrix: np.ndarray) -> dict:
    user_sim = cosine_similarity(matrix)
    np.fill_diagonal(user_sim, 0.0)
    return {"matrix": matrix, "user_sim": user_sim}


def user_cf_score_all_items(model: dict, user_idx: int) -> np.ndarray:
    sims = model["user_sim"][user_idx]
    numerator = sims @ model["matrix"]
    denominator = np.abs(sims) @ (model["matrix"] > 0)
    return np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator > 0)


def user_cf_predict(model: dict, user_idx: int, item_idx: int) -> float:
    return float(user_cf_score_all_items(model, user_idx)[item_idx])


def fit_item_cf(matrix: np.ndarray) -> dict:
    item_sim = cosine_similarity(matrix.T)
    np.fill_diagonal(item_sim, 0.0)
    return {"matrix": matrix, "item_sim": item_sim}


def item_cf_score_all_items(model: dict, user_idx: int) -> np.ndarray:
    user_ratings = model["matrix"][user_idx]
    numerator = user_ratings @ model["item_sim"]
    denominator = (user_ratings > 0).astype(np.float32) @ np.abs(model["item_sim"])
    return np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator > 0)


def item_cf_predict(model: dict, user_idx: int, item_idx: int) -> float:
    return float(item_cf_score_all_items(model, user_idx)[item_idx])


popular_baseline = fit_most_popular(train_df, n_items)
user_cf_baseline = fit_user_cf(train_matrix)
item_cf_baseline = fit_item_cf(train_matrix)

baseline_rows = []
for name, baseline, predict_fn, score_fn in [
    ("most_popular", popular_baseline, popular_predict, popular_score_all_items),
    ("user_cf", user_cf_baseline, user_cf_predict, user_cf_score_all_items),
    ("item_cf", item_cf_baseline, item_cf_predict, item_cf_score_all_items),
]:
    row = {"model": name}
    row["val_rmse"] = rmse_from_predictions(val_df, lambda u, i, m=baseline, f=predict_fn: f(m, u, i))
    row.update(ranking_metrics(lambda u, m=baseline, f=score_fn: f(m, u), val_positive, train_seen, top_k=cfg.k))
    baseline_rows.append(row)

baseline_results = pd.DataFrame(baseline_rows).sort_values(f"ndcg@{cfg.k}", ascending=False)
baseline_results


,model,val_rmse,precision@10,recall@10,map@10,ndcg@10,coverage
0,most_popular,1.064086,0.009533,0.095335,0.030295,0.044926,0.048157
1,user_cf,1.066870,0.000000,0.000000,0.000000,0.000000,0.018430
2,item_cf,1.110777,0.000000,0.000000,0.000000,0.000000,0.175386


In [5]:
class MatrixFactorization(nn.Module):
    def __init__(self, n_users: int, n_items: int, n_factors: int, use_bias: bool = True) -> None:
        super().__init__()
        self.user_factors = nn.Embedding(n_users, n_factors)
        self.item_factors = nn.Embedding(n_items, n_factors)
        self.use_bias = use_bias
        if use_bias:
            self.user_bias = nn.Embedding(n_users, 1)
            self.item_bias = nn.Embedding(n_items, 1)
            self.global_bias = nn.Parameter(torch.zeros(1))
        nn.init.normal_(self.user_factors.weight, std=0.05)
        nn.init.normal_(self.item_factors.weight, std=0.05)

    def forward(self, user_idx: torch.Tensor, item_idx: torch.Tensor) -> torch.Tensor:
        dot = (self.user_factors(user_idx) * self.item_factors(item_idx)).sum(dim=1)
        if self.use_bias:
            dot = (
                dot
                + self.user_bias(user_idx).squeeze(1)
                + self.item_bias(item_idx).squeeze(1)
                + self.global_bias
            )
        return dot

    def predict_all_items(self, user_idx: int) -> np.ndarray:
        self.eval()
        with torch.no_grad():
            user_vector = self.user_factors.weight[user_idx]
            scores = self.item_factors.weight @ user_vector
            if self.use_bias:
                scores = scores + self.item_bias.weight.squeeze(1) + self.user_bias.weight[user_idx, 0] + self.global_bias
            return scores.detach().cpu().numpy()


train_dataset = TensorDataset(
    torch.tensor(train_df["user_idx"].to_numpy(), dtype=torch.long),
    torch.tensor(train_df["item_idx"].to_numpy(), dtype=torch.long),
    torch.tensor(train_df["rating"].to_numpy(), dtype=torch.float32),
)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)


def evaluate_mf_rmse(model: MatrixFactorization, frame: pd.DataFrame) -> float:
    model.eval()
    users = torch.tensor(frame["user_idx"].to_numpy(), dtype=torch.long, device=DEVICE)
    items = torch.tensor(frame["item_idx"].to_numpy(), dtype=torch.long, device=DEVICE)
    targets = frame["rating"].to_numpy(dtype=np.float32)
    with torch.no_grad():
        predictions = model(users, items).detach().cpu().numpy()
    return float(np.sqrt(np.mean((predictions - targets) ** 2)))


def train_one_experiment(name: str, n_factors: int, reg: float, optimizer_name: str, use_bias: bool):
    model = MatrixFactorization(n_users, n_items, n_factors=n_factors, use_bias=use_bias).to(DEVICE)
    optimizer_cls = torch.optim.Adam if optimizer_name == "adam" else torch.optim.SGD
    optimizer = optimizer_cls(model.parameters(), lr=cfg.learning_rate)
    writer = SummaryWriter(f"{cfg.log_root}/{name}")
    fixed_users = sorted(list(val_positive.keys()))[:3]
    history = []
    best_state = None
    best_row = None
    best_val_score = -1.0

    for epoch in range(1, cfg.num_epochs + 1):
        model.train()
        epoch_losses = []
        for user_idx, item_idx, ratings_batch in tqdm(train_loader, leave=False, desc=name + f" epoch {epoch}"):
            user_idx = user_idx.to(DEVICE)
            item_idx = item_idx.to(DEVICE)
            ratings_batch = ratings_batch.to(DEVICE)

            predictions = model(user_idx, item_idx)
            mse = torch.mean((predictions - ratings_batch) ** 2)
            reg_term = reg * (
                model.user_factors(user_idx).pow(2).mean() + model.item_factors(item_idx).pow(2).mean()
            )
            loss = mse + reg_term

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_losses.append(float(loss.item()))

        train_loss = float(np.mean(epoch_losses))
        val_rmse = evaluate_mf_rmse(model, val_df)
        ranking = ranking_metrics(model.predict_all_items, val_positive, train_seen, top_k=cfg.k)

        writer.add_scalar("loss/train", train_loss, epoch)
        writer.add_scalar("rmse/val", val_rmse, epoch)
        for metric_name, metric_value in ranking.items():
            writer.add_scalar(metric_name.replace("@", "_at_"), metric_value, epoch)

        for user_idx in fixed_users:
            scores = model.predict_all_items(user_idx)
            seen = train_seen.get(user_idx, set())
            if seen:
                scores[list(seen)] = -1e9
            top_items = np.argsort(scores)[-cfg.k:][::-1]
            titles = [item_id_to_title[idx_to_item[item_idx]] for item_idx in top_items[:5]]
            writer.add_text(
                f"topk/user_{user_idx}",
                "\n".join(titles),
                epoch,
            )

        row = {
            "experiment": name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_rmse": val_rmse,
        }
        row.update(ranking)
        history.append(row)
        print(name, row)

        val_score = ranking[f"ndcg@{cfg.k}"]
        if val_score > best_val_score:
            best_val_score = val_score
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            best_row = row.copy()

    if best_state is None:
        raise RuntimeError(f"Failed to select best checkpoint for {name}")

    model.load_state_dict(best_state)
    test_rmse = evaluate_mf_rmse(model, test_df)
    test_ranking = ranking_metrics(model.predict_all_items, test_positive, train_seen, top_k=cfg.k)
    writer.add_hparams(
        {
            "n_factors": n_factors,
            "reg": reg,
            "optimizer": optimizer_name,
            "use_bias": int(use_bias),
        },
        {
            "hparam/best_val_rmse": best_row["val_rmse"],
            f"hparam/best_val_ndcg_at_{cfg.k}": best_row[f"ndcg@{cfg.k}"],
            "hparam/test_rmse": test_rmse,
            f"hparam/test_ndcg_at_{cfg.k}": test_ranking[f"ndcg@{cfg.k}"],
        },
    )
    writer.close()
    return model, pd.DataFrame(history), {
        "best_epoch": best_row["epoch"],
        "best_val_rmse": best_row["val_rmse"],
        f"best_val_precision@{cfg.k}": best_row[f"precision@{cfg.k}"],
        f"best_val_recall@{cfg.k}": best_row[f"recall@{cfg.k}"],
        f"best_val_map@{cfg.k}": best_row[f"map@{cfg.k}"],
        f"best_val_ndcg@{cfg.k}": best_row[f"ndcg@{cfg.k}"],
        "best_val_coverage": best_row["coverage"],
        "test_rmse": test_rmse,
        **test_ranking,
    }


experiments = [
    ("mf_f16_reg1e-4_adam_bias0", 16, 1e-4, "adam", False),
    ("mf_f32_reg1e-4_adam_bias1", 32, 1e-4, "adam", True),
    ("mf_f64_reg1e-4_adam_bias1", 64, 1e-4, "adam", True),
    ("mf_f32_reg1e-3_adam_bias1", 32, 1e-3, "adam", True),
    ("mf_f32_reg1e-4_sgd_bias1", 32, 1e-4, "sgd", True),
]


In [6]:
all_histories = []
final_rows = []
best_model = None
best_score = -1.0

for name, n_factors, reg, optimizer_name, use_bias in experiments:
    model, history_df, experiment_metrics = train_one_experiment(
        name=name,
        n_factors=n_factors,
        reg=reg,
        optimizer_name=optimizer_name,
        use_bias=use_bias,
    )
    all_histories.append(history_df)
    final_rows.append({"experiment": name, **experiment_metrics})
    if experiment_metrics[f"best_val_ndcg@{cfg.k}"] > best_score:
        best_score = experiment_metrics[f"best_val_ndcg@{cfg.k}"]
        best_model = model

history_table = pd.concat(all_histories, ignore_index=True)
final_results = pd.DataFrame(final_rows).sort_values(f"best_val_ndcg@{cfg.k}", ascending=False)

display(history_table.tail())
display(baseline_results)
display(final_results)


mf_f16_reg1e-4_adam_bias0 epoch 1:   0%|          | 0/48 [00:00<?, ?it/s]

mf_f16_reg1e-4_adam_bias0 {'experiment': 'mf_f16_reg1e-4_adam_bias0', 'epoch': 1, 'train_loss': 13.593247751394907, 'val_rmse': 3.5585825443267822, 'precision@10': 0.006693711967545639, 'recall@10': 0.06693711967545639, 'map@10': 0.020772561898322547, 'ndcg@10': 0.03115879095969657, 'coverage': 0.1961950059453032}


mf_f16_reg1e-4_adam_bias0 epoch 2:   0%|          | 0/48 [00:00<?, ?it/s]

mf_f16_reg1e-4_adam_bias0 {'experiment': 'mf_f16_reg1e-4_adam_bias0', 'epoch': 2, 'train_loss': 8.560007592042288, 'val_rmse': 2.172788143157959, 'precision@10': 0.008924949290060852, 'recall@10': 0.08924949290060852, 'map@10': 0.03643388389838694, 'ndcg@10': 0.048392969117830184, 'coverage': 0.05826397146254459}


mf_f16_reg1e-4_adam_bias0 epoch 3:   0%|          | 0/48 [00:00<?, ?it/s]

...

mf_f32_reg1e-4_sgd_bias1 epoch 15:   0%|          | 0/48 [00:00<?, ?it/s]

mf_f32_reg1e-4_sgd_bias1 {'experiment': 'mf_f32_reg1e-4_sgd_bias1', 'epoch': 15, 'train_loss': 3.1074650635321936, 'val_rmse': 1.7781888246536255, 'precision@10': 0.0002028397565922921, 'recall@10': 0.002028397565922921, 'map@10': 0.0002535496957403651, 'ndcg@10': 0.0006398881882063464, 'coverage': 0.010107015457788348}


,experiment,epoch,train_loss,val_rmse,precision@10,recall@10,map@10,ndcg@10,coverage
70,mf_f32_reg1e-4_sgd_bias1,11,3.124499,1.780087,0.000203,0.002028,0.000254,0.00064,0.010107
71,mf_f32_reg1e-4_sgd_bias1,12,3.120279,1.779820,0.000203,0.002028,0.000254,0.00064,0.010107
72,mf_f32_reg1e-4_sgd_bias1,13,3.116089,1.779376,0.000203,0.002028,0.000254,0.00064,0.010107
73,mf_f32_reg1e-4_sgd_bias1,14,3.111285,1.778775,0.000203,0.002028,0.000254,0.00064,0.010107
74,mf_f32_reg1e-4_sgd_bias1,15,3.107465,1.778189,0.000203,0.002028,0.000254,0.00064,0.010107


,model,val_rmse,precision@10,recall@10,map@10,ndcg@10,coverage
0,most_popular,1.064086,0.009533,0.095335,0.030295,0.044926,0.048157
1,user_cf,1.066870,0.000000,0.000000,0.000000,0.000000,0.018430
2,item_cf,1.110777,0.000000,0.000000,0.000000,0.000000,0.175386


,experiment,best_epoch,best_val_rmse,best_val_precision@10,best_val_recall@10,best_val_map@10,best_val_ndcg@10,best_val_coverage,test_rmse,precision@10,recall@10,map@10,ndcg@10,coverage
0,mf_f16_reg1e-4_adam_bias0,3,1.226457,0.009939,0.099391,0.037746,0.052168,0.041617,1.237623,0.010905,0.109053,0.035817,0.052833,0.042212
2,mf_f64_reg1e-4_adam_bias1,2,1.388583,0.006897,0.068966,0.021612,0.032304,0.057075,1.380024,0.004733,0.047325,0.015060,0.022511,0.061237
3,mf_f32_reg1e-3_adam_bias1,3,1.198132,0.004665,0.046653,0.019144,0.025527,0.054697,1.236161,0.004938,0.049383,0.016854,0.024198,0.055291
1,mf_f32_reg1e-4_adam_bias1,2,1.609029,0.004868,0.048682,0.016339,0.023845,0.045779,1.606860,0.003909,0.039095,0.010849,0.017202,0.046373
4,mf_f32_reg1e-4_sgd_bias1,1,2.662907,0.000203,0.002028,0.000254,0.000640,0.010107,2.695918,0.000412,0.004115,0.000463,0.001244,0.010702


In [7]:
fixed_user = sorted(list(test_positive.keys()))[0]
scores = best_model.predict_all_items(fixed_user)
seen = train_seen.get(fixed_user, set())
if seen:
    scores[list(seen)] = -1e9
top_items = np.argsort(scores)[-cfg.k:][::-1]

recommendations = pd.DataFrame(
    {
        "item_id": [idx_to_item[item_idx] for item_idx in top_items],
        "title": [item_id_to_title[idx_to_item[item_idx]] for item_idx in top_items],
        "score": [scores[item_idx] for item_idx in top_items],
    }
)
recommendations


,item_id,title,score
0,313,Titanic (1997),3.794209
1,50,Star Wars (1977),3.606062
2,269,"Full Monty, The (1997)",3.500865
3,127,"Godfather, The (1972)",3.449954
4,100,Fargo (1996),3.420297
5,286,"English Patient, The (1996)",3.398190
6,475,Trainspotting (1996),3.329702
7,315,Apt Pupil (1998),3.317785
8,483,Casablanca (1942),3.316638
9,98,"Silence of the Lambs, The (1991)",3.310292


```bash
tensorboard --logdir runs/task8
```